# Snapshot Comparison

Compare two snapshots (default: baseline vs latest) to measure intervention impact on build performance, dependencies, and critical path.

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore", category=FutureWarning)

SNAPSHOTS_DIR = Path("../../data/snapshots")
CONFIG_DIR = Path("../../data/config")

# Parameterise: which snapshots to compare
SNAPSHOT_A = "baseline"
SNAPSHOT_B = "latest"

from buildanalysis.snapshots import SnapshotManager

sm = SnapshotManager(SNAPSHOTS_DIR)

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100, "axes.titlesize": 14})
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
# Resolve snapshot labels
all_snaps = sm.list_snapshots()
print(f"Available snapshots: {len(all_snaps)}")
for snap in all_snaps:
    print(f"  - {snap.label}")

snap_a_meta = next((s for s in all_snaps if s.label.startswith(SNAPSHOT_A)), None)
snap_b_meta = None

if SNAPSHOT_B == "latest":
    snap_b_meta = all_snaps[-1] if all_snaps else None
else:
    snap_b_meta = next((s for s in all_snaps if s.label.startswith(SNAPSHOT_B)), None)

if not snap_a_meta or not snap_b_meta:
    print(f"\nCould not find snapshots: {SNAPSHOT_A} or {SNAPSHOT_B}")
    print("Proceeding with available data...")
else:
    print(f"\nComparing: {snap_a_meta.label} vs {snap_b_meta.label}")

In [ ]:
# Load datasets
if snap_a_meta and snap_b_meta:
    ds_a = sm.load_dataset(snap_a_meta.label)
    ds_b = sm.load_dataset(snap_b_meta.label)

    print(f"Loaded snapshot A: {snap_a_meta.label}")
    print(f"Loaded snapshot B: {snap_b_meta.label}")
else:
    print("Cannot proceed without both snapshots")

## 1. Snapshot Metadata

In [ ]:
if snap_a_meta and snap_b_meta:
    print("\nSNAPSHOT METADATA")
    print("=" * 100)

    metadata_comparison = pd.DataFrame([
        {
            "Property": "Label",
            "Snapshot A (Baseline)": snap_a_meta.label,
            "Snapshot B (Latest)": snap_b_meta.label,
        },
        {
            "Property": "Date",
            "Snapshot A (Baseline)": snap_a_meta.date,
            "Snapshot B (Latest)": snap_b_meta.date,
        },
        {
            "Property": "Git Ref",
            "Snapshot A (Baseline)": snap_a_meta.git_ref,
            "Snapshot B (Latest)": snap_b_meta.git_ref,
        },
        {
            "Property": "Compiler",
            "Snapshot A (Baseline)": snap_a_meta.compiler,
            "Snapshot B (Latest)": snap_b_meta.compiler,
        },
        {
            "Property": "Notes",
            "Snapshot A (Baseline)": snap_a_meta.notes,
            "Snapshot B (Latest)": snap_b_meta.notes,
        },
    ])

    print(metadata_comparison.to_string(index=False))
else:
    print("Cannot display metadata without both snapshots")

## 2. Global Metric Deltas

In [ ]:
if snap_a_meta and snap_b_meta:
    # Compute global metric deltas
    target_metrics_a = ds_a.target_metrics
    target_metrics_b = ds_b.target_metrics

    global_metrics = {
        "total_build_time_ms": (target_metrics_a["total_build_time_ms"].sum(), target_metrics_b["total_build_time_ms"].sum()),
        "target_count": (len(target_metrics_a), len(target_metrics_b)),
        "total_sloc": (target_metrics_a["code_lines_total"].sum() if "code_lines_total" in target_metrics_a.columns else 0,
                       target_metrics_b["code_lines_total"].sum() if "code_lines_total" in target_metrics_b.columns else 0),
    }

    deltas = []
    for metric_name, (val_a, val_b) in global_metrics.items():
        if val_a > 0:
            delta_pct = 100 * (val_b - val_a) / val_a
            delta_abs = val_b - val_a
            status = "improved" if delta_pct < 0 else "regressed" if delta_pct > 0 else "unchanged"
            deltas.append({
                "metric": metric_name,
                "baseline": val_a,
                "latest": val_b,
                "delta_abs": delta_abs,
                "delta_pct": delta_pct,
                "status": status,
            })

    delta_df = pd.DataFrame(deltas)

    print("\nGLOBAL METRIC DELTAS")
    print("=" * 100)
    print(delta_df.to_string(index=False))

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Delta percentage
    colors = ["#2ca02c" if x < 0 else "#d62728" for x in delta_df["delta_pct"]]
    axes[0].barh(range(len(delta_df)), delta_df["delta_pct"], color=colors)
    axes[0].set_yticks(range(len(delta_df)))
    axes[0].set_yticklabels(delta_df["metric"])
    axes[0].axvline(x=0, color="black", linestyle="-", linewidth=0.8)
    axes[0].set_xlabel("Change (%)")
    axes[0].set_title("Global Metric Deltas (%)")

    # Absolute metrics
    x = np.arange(len(delta_df))
    width = 0.35
    axes[1].bar(x - width/2, delta_df["baseline"], width, label="Baseline", alpha=0.8)
    axes[1].bar(x + width/2, delta_df["latest"], width, label="Latest", alpha=0.8)
    axes[1].set_ylabel("Value")
    axes[1].set_title("Absolute Metrics")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(delta_df["metric"], rotation=45, ha="right")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Cannot compute global deltas without both snapshots")

## 3. Target-Level Deltas

In [ ]:
if snap_a_meta and snap_b_meta:
    # Merge targets and compute deltas
    target_comparison = target_metrics_a[["cmake_target", "total_build_time_ms"]].rename(columns={"total_build_time_ms": "time_a"})
    target_comparison = target_comparison.merge(
        target_metrics_b[["cmake_target", "total_build_time_ms"]].rename(columns={"total_build_time_ms": "time_b"}),
        on="cmake_target",
        how="outer"
    ).fillna(0)

    target_comparison["delta_ms"] = target_comparison["time_b"] - target_comparison["time_a"]
    target_comparison["delta_pct"] = np.where(
        target_comparison["time_a"] > 0,
        100 * target_comparison["delta_ms"] / target_comparison["time_a"],
        0
    )

    improvers = target_comparison[target_comparison["delta_ms"] < 0].nsmallest(20, "delta_ms")
    regressors = target_comparison[target_comparison["delta_ms"] > 0].nlargest(20, "delta_ms")

    print("\nTARGET-LEVEL DELTAS")
    print("=" * 100)
    print(f"\nTop 20 Improvers (fastest improvements):")
    print(improvers[["cmake_target", "time_a", "time_b", "delta_ms", "delta_pct"]].to_string(index=False))

    print(f"\nTop 20 Regressors (slowest regressions):")
    print(regressors[["cmake_target", "time_a", "time_b", "delta_ms", "delta_pct"]].to_string(index=False))

    # New targets
    new_targets = target_comparison[target_comparison["time_a"] == 0]
    removed_targets = target_comparison[target_comparison["time_b"] == 0]
    print(f"\nNew targets: {len(new_targets)}")
    print(f"Removed targets: {len(removed_targets)}")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    top_changes = pd.concat([improvers.head(10), regressors.head(10)]).sort_values("delta_ms")
    colors = ["#2ca02c" if x < 0 else "#d62728" for x in top_changes["delta_ms"]]
    ax.barh(range(len(top_changes)), top_changes["delta_ms"], color=colors)
    ax.set_yticks(range(len(top_changes)))
    ax.set_yticklabels(top_changes["cmake_target"], fontsize=9)
    ax.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
    ax.set_xlabel("Build Time Delta (ms)")
    ax.set_title("Top 10 Improvers & Regressors")
    plt.tight_layout()
    plt.show()
else:
    print("Cannot compute target deltas without both snapshots")

## 4. Dependency Changes

In [ ]:
if snap_a_meta and snap_b_meta:
    edge_list_a = ds_a.edge_list
    edge_list_b = ds_b.edge_list

    # Convert to frozenset for comparison
    edges_a = set((row["source_target"], row["dest_target"]) for _, row in edge_list_a.iterrows())
    edges_b = set((row["source_target"], row["dest_target"]) for _, row in edge_list_b.iterrows())

    added_edges = edges_b - edges_a
    removed_edges = edges_a - edges_b
    unchanged_edges = edges_a & edges_b

    print("\nDEPENDENCY CHANGES")
    print("=" * 80)
    print(f"Added edges: {len(added_edges)}")
    print(f"Removed edges: {len(removed_edges)}")
    print(f"Unchanged edges: {len(unchanged_edges)}")
    print(f"Total change: {len(added_edges) - len(removed_edges):+d} edges")

    if len(added_edges) > 0:
        print(f"\nSample added edges (first 10):")
        for i, (src, dst) in enumerate(sorted(added_edges)[:10]):
            print(f"  {src} -> {dst}")

    if len(removed_edges) > 0:
        print(f"\nSample removed edges (first 10):")
        for i, (src, dst) in enumerate(sorted(removed_edges)[:10]):
            print(f"  {src} -> {dst}")
else:
    print("Cannot compute dependency changes without both snapshots")

## 5. Critical Path Comparison

In [ ]:
if snap_a_meta and snap_b_meta:
    import networkx as nx

    # Build graphs
    G_a = nx.DiGraph()
    for _, row in target_metrics_a.iterrows():
        G_a.add_node(row["cmake_target"], build_time=row["total_build_time_ms"])
    for _, row in edge_list_a.iterrows():
        G_a.add_edge(row["source_target"], row["dest_target"])

    G_b = nx.DiGraph()
    for _, row in target_metrics_b.iterrows():
        G_b.add_node(row["cmake_target"], build_time=row["total_build_time_ms"])
    for _, row in edge_list_b.iterrows():
        G_b.add_edge(row["source_target"], row["dest_target"])

    # Compute critical paths
    try:
        cp_a = nx.dag_longest_path_length(G_a, weight="build_time", default_weight=0)
    except:
        cp_a = None

    try:
        cp_b = nx.dag_longest_path_length(G_b, weight="build_time", default_weight=0)
    except:
        cp_b = None

    print("\nCRITICAL PATH COMPARISON")
    print("=" * 80)
    if cp_a is not None and cp_b is not None:
        delta = cp_b - cp_a
        delta_pct = 100 * delta / cp_a if cp_a > 0 else 0
        print(f"Baseline critical path: {cp_a:.0f} ms")
        print(f"Latest critical path: {cp_b:.0f} ms")
        print(f"Delta: {delta:+.0f} ms ({delta_pct:+.1f}%)")
        if delta < 0:
            print(f"\n✓ Critical path improved by {abs(delta):.0f} ms")
        elif delta > 0:
            print(f"\n✗ Critical path regressed by {delta:.0f} ms")
    else:
        print("Could not compute critical paths (possible cycles in dependency graph)")
else:
    print("Cannot compute critical path without both snapshots")

## 6. Intervention Effectiveness

In [ ]:
if snap_a_meta and snap_b_meta:
    interventions = snap_b_meta.interventions_applied if hasattr(snap_b_meta, "interventions_applied") else []

    print("\nINTERVENTION EFFECTIVENESS")
    print("=" * 80)

    if isinstance(interventions, list) and len(interventions) > 0:
        print(f"Interventions applied in {snap_b_meta.label}:")
        for intervention in interventions:
            print(f"  - {intervention}")

        print(f"\nObserved improvements:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] < 0:
                print(f"  ✓ {row['metric']}: {row['delta_pct']:.1f}% improvement")

        print(f"\nObserved regressions:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] > 0:
                print(f"  ✗ {row['metric']}: {row['delta_pct']:.1f}% regression")
    else:
        print("No interventions recorded for this snapshot.")
        print("\nOverall metric changes:")
        for idx, row in delta_df.iterrows():
            if row["delta_pct"] < 0:
                print(f"  ✓ {row['metric']}: {row['delta_pct']:.1f}% improvement")
            elif row["delta_pct"] > 0:
                print(f"  ✗ {row['metric']}: {row['delta_pct']:.1f}% regression")
else:
    print("Cannot assess intervention effectiveness without both snapshots")

## 7. Summary

In [ ]:
if snap_a_meta and snap_b_meta:
    print("\n" + "=" * 80)
    print("SNAPSHOT COMPARISON SUMMARY")
    print("=" * 80)

    # Overall assessment
    improved_count = sum(1 for _, row in delta_df.iterrows() if row["delta_pct"] < 0)
    regressed_count = sum(1 for _, row in delta_df.iterrows() if row["delta_pct"] > 0)

    if improved_count > regressed_count:
        print(f"\n✓ BUILD IMPROVED: {improved_count} metrics improved, {regressed_count} regressed")
    elif regressed_count > improved_count:
        print(f"\n✗ BUILD REGRESSED: {regressed_count} metrics regressed, {improved_count} improved")
    else:
        print(f"\n= NO CLEAR CHANGE: {improved_count} metrics improved, {regressed_count} regressed")

    # Specific callouts
    build_time_delta = delta_df[delta_df["metric"] == "total_build_time_ms"]["delta_pct"].values
    if len(build_time_delta) > 0:
        build_pct = build_time_delta[0]
        if build_pct < 0:
            print(f"\nBuild time improved by {abs(build_pct):.1f}%")
        elif build_pct > 0:
            print(f"\nBuild time regressed by {build_pct:.1f}%")

    print(f"\nComparison: {snap_a_meta.label} → {snap_b_meta.label}")
    print("=" * 80)
else:
    print("Cannot generate summary without both snapshots")